# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np


df = pd.read_csv("electricity-prod-source-stacked.csv")


df = df.melt(
    id_vars=['Entity', 'Code', 'Year'],
    value_vars=[
        'Other renewables', 'Bioenergy', 'Solar', 'Wind',
        'Hydropower', 'Nuclear', 'Oil', 'Gas', 'Coal'
    ],
    var_name='Source',
    value_name='TWh'
)

df['Source'] = df['Source'].replace({
    'Hydropower': 'Hydro',
    'Gas': 'Natural Gas'
})

source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable',
    'Other renewables': 'Renewable', 'Bioenergy': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 61911 rows
          Entity Code  Year            Source    TWh Source_Type
0  ASEAN (Ember)  NaN  2000  Other renewables  16.93   Renewable
1  ASEAN (Ember)  NaN  2001  Other renewables  16.40   Renewable
2  ASEAN (Ember)  NaN  2002  Other renewables  16.61   Renewable
3  ASEAN (Ember)  NaN  2003  Other renewables  15.74   Renewable
4  ASEAN (Ember)  NaN  2004  Other renewables  16.57   Renewable
5  ASEAN (Ember)  NaN  2005  Other renewables  16.50   Renewable
6  ASEAN (Ember)  NaN  2006  Other renewables  17.13   Renewable
7  ASEAN (Ember)  NaN  2007  Other renewables  17.24   Renewable
8  ASEAN (Ember)  NaN  2008  Other renewables  19.03   Renewable
9  ASEAN (Ember)  NaN  2009  Other renewables  19.62   Renewable


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1
# YOUR CODE HERE
# Task 1 — Treemap: fossil fuel dependency by country
import pandas as pd
import plotly.express as px

# Filter to fossil sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil']

# Build treemap
fig = px.treemap(
    df_fossil,
    path=['Entity', 'Source'],  # hierarchy: Entity → Source
    values='TWh',                          # show TWh values
    color='Source',                        # encode fossil source type
    color_discrete_map={
        'Coal': '#8B0000',        # dark red
        'Oil': '#FF8C00',         # orange
        'Natural Gas': '#4682B4'  # steel blue
    },
    title='Fossil Fuel Dependency by Country'
)

# Grey out parent nodes (Region and Country level)
fig.update_traces(root_color='lightgrey')

# Display the treemap
fig.show()

# Optional insight
most_fossil_dependent = (
    df_fossil.groupby('Entity')['TWh'].sum().idxmax()
)
print(f"The most fossil-dependent country is {most_fossil_dependent}.")


The most fossil-dependent country is World.


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [3]:

import plotly.express as px


tips = px.data.tips()


tips_sum = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()


fig = px.sunburst(
    tips_sum,
    path=['day', 'time', 'smoker'],       
    values='total_bill',                  
    color='smoker',                       
    color_discrete_map={'Yes': '#E69F00', 'No': '#56B4E9'},  
    title='Tipping Behaviour by Day and Meal Time'
)


fig.update_traces(root_color='lightgrey')


fig.update_traces(textinfo='label+percent parent')


fig.show()

most_spending = (
    tips_sum.groupby(['day', 'time'])['total_bill'].sum().idxmax()
)
print(f"Most spending happens on {most_spending[0]} during {most_spending[1]}.")


Most spending happens on Sat during Dinner.


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [4]:
# Task 3 — charts
# YOUR CODE HERE
# Task 3 — Treemap vs Bar: low-carbon energy by country
# Task 3 — Treemap vs Bar: low-carbon energy by country
import pandas as pd
import plotly.express as px

# Filter to low-carbon sources only
df_low = df.loc[df['Source_Type'] == 'Low-carbon']

# Aggregate TWh by country (Entity)
df_low_sum = df_low.groupby('Entity')['TWh'].sum().reset_index()

# ✅ Add a dummy column for the root node
df_low_sum['All'] = 'Low-carbon'

# Treemap: single-level with dummy root node
fig_treemap = px.treemap(
    df_low_sum,
    path=['All', 'Entity'],  # use Entity as country
    values='TWh',
    color='TWh',
    color_continuous_scale='Viridis',
    title='Low-carbon Energy by Country (Treemap)'
)
fig_treemap.update_traces(root_color='lightgrey')
fig_treemap.show()

# Bar chart: sorted by TWh, horizontal orientation
df_low_sum_sorted = df_low_sum.sort_values('TWh', ascending=True)
fig_bar = px.bar(
    df_low_sum_sorted,
    x='TWh',
    y='Entity',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon Energy by Country (Bar Chart)'
)
fig_bar.show()

# Insight: leading country
leading_country = df_low_sum_sorted.iloc[-1]['Entity']
print(f"The leading country in low-carbon energy generation is {leading_country}.")


The leading country in low-carbon energy generation is World.
